In [ ]:
def main(datasource, start_date, end_date):
    """
    factor function

    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format

    Returns:
        pd.DataFrame: Factor data with columns ['date', 'instrument', 'factor']
    """
    import pandas as pd
    import dai

    # This function demonstrates how to:
    # 1. Query market from the specified datasource
    # 2. Calculate a simple factor using SQL aggregation
    # 3. Return properly formatted factor data for analysis

    # 因子公式 https://bigquant.com/wiki/doc/Rceb2JQBdS
    # This represents the volume-weighted average price (VWAP): sum(amount) / sum(volume)
    sql = f"""
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            -1 * SUM(amount) / SUM(volume) AS factor
        FROM {datasource}
        GROUP BY date::DATE, instrument
    """

    # Data is queried with a 7-day lookback buffer for rolling calculations
    lookback_days = 7
    query_start_date = pd.to_datetime(start_date) - pd.Timedelta(days=lookback_days)
    # compression=True 降低 instrument 字符串字段的内存占用
    df = dai.query(sql, filters={'date': [query_start_date, end_date]}, compression=True).df()

    # Final output is filtered to the exact date range requested
    df = df[df['date'].between(start_date, end_date)]

    return df


if __name__ == '__main__':
    """
    FOR Development in __main__

    This section demonstrates how to:
    1. Calculate factors using the main() function
    2. Visualize and analyze results using the factorlens module

    Tips:
        - Adjust the date range to test different time periods
        - Modify the SQL query in main() to create your own factors
        - Use factorlens to evaluate factor performance metrics
    """
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()

    datasource = 'bigalpha_factor_2026_stock_bar1m'
    start_date = '2024-01-01 00:00:00'
    end_date = '2025-12-31 23:59:59'

    logger.info(f"Calculating factor for period: {start_date} to {end_date}")
    factor_data = main(datasource, start_date, end_date)

    import dai
    logger.info(f"Getting factor library: {start_date} to {end_date}")
    sql = "SELECT * FROM bigalpha_factor_2026_factorlib"
    factor_pool = dai.query(sql, filters={'date': [start_date, end_date]}).df()

    result = M.bigalpha_factorminer.v9999(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )
